# guard_eval на Kaggle (2× T4)

Динамическое квантование guardrail-моделей: качество и смены вердиктов на `AutotuneGuardrails/tails`, скорость по бакетам, сводка. Ноутбук ничего не отправляет в git: все результаты собираются в архив для скачивания.

**Перед запуском:**
1. Settings → Accelerator: **GPU T4 x2**, Internet: **On**.
2. Add-ons → Secrets: `HF_TOKEN` (доступ к `AutotuneGuardrails/tails`), привязать к ноутбуку.
3. Код: подключить приватный Kaggle Dataset с архивом `guard-eval-code.zip` (Add Input) **или** оставить клонирование ветки с GitHub, если она там есть.
4. Заполнить ячейку настроек.

Полный прогон — несколько часов: удобнее **Save Version → Save & Run All (Commit)**, тогда он идёт в фоне до 12 ч. В конце скачать из Output `guard_eval_results.zip`.

На T4: точность fp16 (bf16 нет), `torchao_fp8_dyn` будет помечен «не поддерживается» (нужен sm_89+), WildGuard не входит в список по умолчанию — эталон 7B в fp16 не помещается в 15 ГБ.

## 1. Настройки

In [1]:
CODE_INPUT = "/kaggle/input"   # где искать подключённый Kaggle Dataset с кодом (guard-eval-code.zip); если не найден — клонируем ветку
REPO_URL = "https://github.com/kekwak/HiveTrace-AutoTune-Guardrails.git"
BRANCH = "quality/guard-eval"
MODELS = "gliner-guard koala hivetrace-guard-pro openguardrails"
METHODS = "torchao_int8_dyn torchao_fp8_dyn bnb_int8"
EXTRA_ARGS = ""          # например "--limit-per-split 20" для быстрой проверки всей цепочки
TWO_GPUS = True          # качество параллельно на двух T4, скорость — по очереди на одной
TORCHAO = "torchao"      # если torchao не совместим с torch на Kaggle — закрепить версию, напр. "torchao==0.14.1"

WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/guard-eval"

## 2. Токен Hugging Face из Kaggle Secrets

Значение не печатается и не попадает в ноутбук.

In [2]:
import os
from kaggle_secrets import UserSecretsClient

try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN: получен")
except Exception as e:
    raise RuntimeError("Секрет HF_TOKEN не найден или не привязан к ноутбуку") from e
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

HF_TOKEN: получен


## 3. Код и зависимости

Код берётся из подключённого Kaggle Dataset (ищется по всему `/kaggle/input`: zip или уже распакованная папка с `guard_eval/`), а если его нет — клонируется ветка с GitHub. Если код уже скопирован в `/kaggle/working/guard-eval` в этой сессии, он не перезаписывается: чтобы взять новую версию, удалите эту папку.

In [3]:
import glob, os, shutil, subprocess, zipfile

unpacked = glob.glob(f"{CODE_INPUT}/**/guard_eval/__main__.py", recursive=True)
archives = [a for a in glob.glob(f"{CODE_INPUT}/**/*.zip", recursive=True) if "guard_eval/__main__.py" in zipfile.ZipFile(a).namelist()]
if os.path.isdir(REPO_DIR):
    print("код уже на месте:", REPO_DIR)
elif unpacked:                       # Kaggle распаковал архив при загрузке датасета
    shutil.copytree(os.path.dirname(os.path.dirname(unpacked[0])), REPO_DIR)
    print("код взят из Kaggle Dataset:", os.path.dirname(os.path.dirname(unpacked[0])))
elif archives:                       # Kaggle оставил zip как есть
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(REPO_DIR)
    print("код взят из архива:", archives[0])
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
    print("Kaggle Dataset с кодом не найден — склонирована ветка", BRANCH)
assert os.path.isfile(f"{REPO_DIR}/guard_eval/__main__.py"), "в коде нет guard_eval — проверьте архив"
for f in glob.glob(f"{REPO_DIR}/scripts/*.sh"):
    os.chmod(f, 0o755)

код уже на месте: /kaggle/working/guard-eval


In [4]:
!pip install -q -r {REPO_DIR}/requirements.txt "{TORCHAO}" 2>&1 | grep -vE "dependency conflicts|requires|incompatible" | tail -5
!python -c "import torch, transformers, torchao, bitsandbytes, gliner2; print('torch', torch.__version__, '| transformers', transformers.__version__, '| torchao', torchao.__version__, '| bitsandbytes', bitsandbytes.__version__)"

torch 2.11.0+cu128 | transformers 5.20.1 | torchao 0.10.0 | bitsandbytes 0.50.2


## 4. Проверка GPU и методов квантования

Ожидаемо на T4: `torchao_fp8_dyn` — не поддерживается. Если `torchao_int8_dyn` или `bnb_int8` падают в smoke-тесте, их прогоны тоже завершатся ошибкой — тогда стоит убрать их из `METHODS` или закрепить версию библиотеки.

In [5]:
!cd {REPO_DIR} && python -m guard_eval check

{
  "python": "3.13.15",
  "cpu_count": 4,
  "device": "cuda:0",
  "gpu": "Tesla T4",
  "capability": "sm_75",
  "gpu_memory_gb": 14.6,
  "cuda_runtime": "12.8",
  "driver": "580.178.04",
  "device_class": "Tesla T4 (sm_75)"
}
пакеты: {"torch": "2.11.0+cu128", "transformers": "5.20.1", "torchao": "0.10.0", "bitsandbytes": "0.50.2", "gliner2": "2.1.0", "numpy": "2.1.3", "pandas": "2.3.3", "huggingface_hub": "1.33.0"}
torchao_int8_dyn   поддерживается
torchao_fp8_dyn    нет: FP8 требует sm_89+, у GPU sm_75
bnb_int8           поддерживается
smoke на маленькой сети (torch.float16): {
  "torchao_int8_dyn": {
    "ok": true,
    "relative_error": 0.01509
  },
  "torchao_fp8_dyn": {
    "ok": false,
    "error": "UnsupportedQuant: FP8 требует sm_89+, у GPU sm_75"
  },
  "bnb_int8": {
    "ok": true,
    "relative_error": 0.01521
  }
}


## 5. Прогон: все модели × все методы

Вывод по каждой модели — в `logs/` внутри репозитория. Повторный запуск продолжает с места остановки: готовые тексты берутся из кэша.

In [9]:
script = "scripts/run_all_2gpu.sh" if TWO_GPUS else "scripts/run_all.sh"
!cd {REPO_DIR} && MODELS="{MODELS}" METHODS="{METHODS}" bash {script} {EXTRA_ARGS}

{
  "python": "3.13.15",
  "cpu_count": 4,
  "device": "cuda:0",
  "gpu": "Tesla T4",
  "capability": "sm_75",
  "gpu_memory_gb": 14.6,
  "cuda_runtime": "12.8",
  "driver": "580.178.04",
  "device_class": "Tesla T4 (sm_75)"
}
пакеты: {"torch": "2.11.0+cu128", "transformers": "5.20.1", "torchao": "0.10.0", "bitsandbytes": "0.50.2", "gliner2": "2.1.0", "numpy": "2.1.3", "pandas": "2.3.3", "huggingface_hub": "1.33.0"}
torchao_int8_dyn   поддерживается
torchao_fp8_dyn    нет: FP8 требует sm_89+, у GPU sm_75
bnb_int8           поддерживается
smoke на маленькой сети (torch.float16): {
  "torchao_int8_dyn": {
    "ok": true,
    "relative_error": 0.01509
  },
  "torchao_fp8_dyn": {
    "ok": false,
    "error": "UnsupportedQuant: FP8 требует sm_89+, у GPU sm_75"
  },
  "bnb_int8": {
    "ok": true,
    "relative_error": 0.01521
  }
}
=== Фаза 1: качество. GPU 0: gliner-guard hivetrace-guard-pro; GPU 1: koala openguardrails
[GPU 0] качество gliner-guard: код 1
[GPU 1] качество koala: код 1
[G

## 6. Сводка

In [10]:
from IPython.display import Markdown, display

summary = f"{REPO_DIR}/reports/SUMMARY.md"
display(Markdown(open(summary).read()) if os.path.exists(summary) else "сводки нет — смотрите logs/")

# Сводка сравнений

| модель | A | B | N | смен | хуже/лучше | macro смен | macro ΔFNR | macro ΔFPR | веса A→B, МБ | p95 ↓ varying_short | p95 ↓ varying_medium | p95 ↓ varying_long | p95 ↓ mixed |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| gliner-guard | gliner-guard__none__float16 | gliner-guard__bnb_int8__float16 | 14249 | 150 (1.1% [0.9%; 1.2%]) | 78/72 | 1.0% | 0.5% | -0.9% | 586→481 | -242.2% | -248.0% | -231.3% | -245.0% |
| gliner-guard | gliner-guard__none__float16 | gliner-guard__torchao_int8_dyn__float16 | 14249 | 215 (1.5% [1.3%; 1.7%]) | 106/109 | 1.2% | 0.4% | -0.7% | 586→586 | -744.6% | -762.5% | -690.2% | -692.1% |
| hivetrace-guard-pro | hivetrace-guard-pro__none__float16 | hivetrace-guard-pro__bnb_int8__float16 | 14249 | 297 (2.1% [1.9%; 2.3%]) | 163/134 | 2.2% | -0.0% | 0.0% | 1137→717 | -155.7% | -113.7% | -57.0% | -56.2% |
| hivetrace-guard-pro | hivetrace-guard-pro__none__float16 | hivetrace-guard-pro__torchao_int8_dyn__float16 | 14249 | 564 (4.0% [3.7%; 4.3%]) | 376/188 | 4.0% | 2.6% | -0.2% | 1137→1137 | -388.8% | -293.5% | -155.1% | -154.1% |
| koala | koala__none__float16 | koala__bnb_int8__float16 | 14249 | 6 (0.0% [0.0%; 0.1%]) | 6/0 | 0.0% | 0.0% | 0.0% | 266→171 | -182.1% | -172.6% | -174.0% | -189.3% |
| koala | koala__none__float16 | koala__torchao_int8_dyn__float16 | 14249 | 10 (0.1% [0.0%; 0.1%]) | 8/2 | 0.1% | 0.0% | 0.0% | 266→266 | -562.8% | -534.0% | -511.1% | -525.9% |
| openguardrails | openguardrails__none__float16 | openguardrails__bnb_int8__float16 | 14249 | 278 (2.0% [1.7%; 2.2%]) | 120/158 | 2.2% | -0.9% | 0.3% | 7672→4207 | -25.1% | -19.2% | -1.9% | -2.4% |
| openguardrails | openguardrails__none__float16 | openguardrails__torchao_int8_dyn__float16 | 14249 | 374 (2.6% [2.4%; 2.9%]) | 168/206 | 3.0% | -1.6% | 1.1% | 7672→7672 | -193.7% | -167.4% | -98.4% | -99.3% |

смен — изменённые вердикты B относительно A (pooled, 95% bootstrap CI); p95 ↓ — снижение p95 полного запроса.


## 7. Архив результатов для скачивания

`/kaggle/working/guard_eval_results.zip` в Output:
- `results/quality/runs/` — прогоны (скоры по `text_hash`, манифесты, скорость), их потом можно добавить в репозиторий;
- `reports/` — отчёты и `SUMMARY.md` (есть тексты из закрытых наборов — не публиковать);
- `logs/` — логи.

In [7]:
import os, shutil, tempfile

with tempfile.TemporaryDirectory() as tmp:
    for name in ("results/quality/runs", "reports", "logs"):
        if os.path.isdir(f"{REPO_DIR}/{name}"):
            shutil.copytree(f"{REPO_DIR}/{name}", f"{tmp}/{name}")
    archive = shutil.make_archive(f"{WORK}/guard_eval_results", "zip", tmp)
print(archive, f"{os.path.getsize(archive) / 2**20:.1f} МБ")

/kaggle/working/guard_eval_results.zip 0.1 МБ


In [8]:
import os, shutil, tempfile, time
from datetime import datetime, timezone
from huggingface_hub import HfApi

REPO_ID = "AutotuneGuardrails/eval-runs"
REPO_DIR = globals().get("REPO_DIR", "/kaggle/working/guard-eval")
WORK = globals().get("WORK", "/kaggle/working")

# 1. архив: results/quality/runs + reports + logs (заново, чтобы включить всё на момент запуска)
with tempfile.TemporaryDirectory() as tmp:
    for name in ("results/quality/runs", "reports", "logs"):
        if os.path.isdir(f"{REPO_DIR}/{name}"):
            shutil.copytree(f"{REPO_DIR}/{name}", f"{tmp}/{name}")
    archive = shutil.make_archive(f"{WORK}/guard_eval_results", "zip", tmp)
print(f"архив: {archive}, {os.path.getsize(archive) / 2**20:.1f} МБ")

# 2. репозиторий: создать приватным или проверить, что существующий приватный
api = HfApi()
api.create_repo(REPO_ID, repo_type="dataset", private=True, exist_ok=True)
if not api.dataset_info(REPO_ID).private:
    raise RuntimeError(f"{REPO_ID} публичный — загрузка отменена (в отчётах тексты из закрытых наборов)")

# 3. загрузка с повторами; в имени — время UTC, чтобы прогоны не перезаписывали друг друга
stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
target = f"kaggle_t4/{stamp}/guard_eval_results.zip"
for attempt in range(1, 4):
    try:
        api.upload_file(path_or_fileobj=archive, path_in_repo=target, repo_id=REPO_ID, repo_type="dataset",
                        commit_message=f"guard_eval results from Kaggle T4, {stamp}")
        print(f"загружено: https://huggingface.co/datasets/{REPO_ID}/blob/main/{target}")
        break
    except Exception as e:
        print(f"попытка {attempt} не удалась: {type(e).__name__}: {e}")
        if attempt == 3:
            raise
        time.sleep(30 * attempt)

архив: /kaggle/working/guard_eval_results.zip, 0.1 МБ
загружено: https://huggingface.co/datasets/AutotuneGuardrails/eval-runs/blob/main/kaggle_t4/20261105T071324Z/guard_eval_results.zip


## 8. Профилирование: откуда задержки квантованных моделей

Запускать после раздела 5 (веса уже в кэше, GPU свободны). Работает в ядре ноутбука на `cuda:0`; если ядро перезапускалось — сначала выполнить ячейки 1–3.

Два уровня:
1. **Один слой `nn.Linear`** формы как в модели: fp16 против torchao / bnb, в eager и с `torch.compile`. Показывает, сколько стоит квантованный слой сам по себе и что внутри (ядра, синхронизации, операции на CPU).
2. **Целая модель, один запрос** (батч 1, фиксированный текст ~128 токенов): время без профайлера, время ядер на GPU, доля занятости GPU, число запусков ядер и синхронизаций CPU↔GPU на запрос, топ операций по времени CPU и GPU.

Если у квантованной версии GPU занята мало, а синхронизаций или CPU-операций много — тормозит обвязка, а не умножение. Если GPU занята, но ядра медленные — медленный INT8 GEMM на Turing.

Результаты: таблицы в выводе и `logs/profile/` (тексты и трейсы для https://ui.perfetto.dev). После раздела снова выполните раздел 7 (и загрузку в HF, если она есть), чтобы профили попали в архив.

In [6]:
PROFILE_MODELS = ["koala", "hivetrace-guard-pro"]       # маленькие и быстрые; openguardrails можно добавить, но дольше
PROFILE_METHODS = ["none", "torchao_int8_dyn", "bnb_int8"]
TRY_COMPILE = True     # дополнительно torchao_int8_dyn с torch.compile (первая компиляция — минуты)
N_REQUESTS, WARMUP = 20, 10

import gc, json, os, sys, time
import pandas as pd
import torch
from torch import nn
from torch.profiler import ProfilerActivity, profile

sys.path.insert(0, REPO_DIR)
from guard_eval import quant
from guard_eval.adapters import make_adapter
from guard_eval.spec import load_spec, resolve_dtype, with_quant
from guard_eval.speed import workloads

DEVICE = "cuda:0"
PROF_DIR = f"{REPO_DIR}/logs/profile"
os.makedirs(PROF_DIR, exist_ok=True)
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)


def trace_stats(trace_path, n):
    """Сводка по chrome-трейсу на один вызов: время ядер, запуски, синхронизации, копирования."""
    events = json.load(open(trace_path))["traceEvents"]
    kernels = [e for e in events if e.get("cat") == "kernel"]
    runtime = [e for e in events if e.get("cat") == "cuda_runtime"]
    cpu_ops = [e for e in events if e.get("cat") == "cpu_op"]
    count = lambda pred: sum(1 for e in runtime if pred(e["name"])) / n
    return {
        "gpu_kernel_ms": sum(e["dur"] for e in kernels) / 1e3 / n,
        "kernels": len(kernels) / n,
        "syncs": count(lambda s: "Synchronize" in s),
        "memcpy": count(lambda s: s.startswith("cudaMemcpy")),
        "cpu_ops": len(cpu_ops) / n,
    }


def top_table(prof, by, rows=12):
    try:
        return prof.key_averages().table(sort_by=by, row_limit=rows)
    except Exception:   # в старых версиях torch ключ называется self_cuda_time_total
        return prof.key_averages().table(sort_by=by.replace("device", "cuda"), row_limit=rows)


def free():
    gc.collect()
    torch.cuda.empty_cache()

Tesla T4 | torch 2.11.0+cu128


### 8.1. Один квантованный слой

In [7]:
def layer_case(method, k, n_out, m, compiled, iters=200):
    torch.manual_seed(0)
    layer = nn.Sequential(nn.Linear(k, n_out)).to(torch.float16)
    quant.quantize(layer, method, DEVICE, skip=())
    fn = torch.compile(layer, dynamic=False) if compiled else layer
    x = torch.randn(m, k, dtype=torch.float16, device=DEVICE)
    with torch.inference_mode():
        for _ in range(20):
            fn(x)
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(iters):
            fn(x)
        torch.cuda.synchronize()
        ms = (time.perf_counter() - t0) * 1e3 / iters
        with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
            for _ in range(20):
                fn(x)
            torch.cuda.synchronize()
    label = f"linear_{k}x{n_out}_m{m}__{method}{'__compile' if compiled else ''}"
    trace = f"{PROF_DIR}/{label}.json"
    prof.export_chrome_trace(trace)
    (open(f"{PROF_DIR}/{label}.txt", "w")).write(top_table(prof, "self_cpu_time_total") + "\n" + top_table(prof, "self_device_time_total"))
    return {"слой": f"{k}→{n_out}, M={m}", "метод": method + (" + compile" if compiled else ""), "мс на вызов": round(ms, 3),
            **{k2: round(v, 3) for k2, v in trace_stats(trace, 20).items()}}, prof


SHAPES = [(768, 768, 128), (1024, 3072, 512)]   # слой KoalaAI на коротком входе; MLP HiveTraceGuard на типичном
layer_rows, layer_profiles = [], {}
for k, n_out, m in SHAPES:
    for method in PROFILE_METHODS:
        for compiled in ([False, True] if TRY_COMPILE and method != "bnb_int8" else [False]):
            try:
                row, prof = layer_case(method, k, n_out, m, compiled)
                layer_rows.append(row)
                layer_profiles[(k, method, compiled)] = prof
            except Exception as e:
                layer_rows.append({"слой": f"{k}→{n_out}, M={m}", "метод": method + (" + compile" if compiled else ""),
                                   "мс на вызов": None, "ошибка": f"{type(e).__name__}: {e}"[:200]})
            free()

layer_df = pd.DataFrame(layer_rows)
layer_df.to_csv(f"{PROF_DIR}/layers.csv", index=False)
display(layer_df)

/usr/local/lib/python3.13/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(
W1105 06:06:44.619000 115 torch/_inductor/utils.py:1731] [0/1] Not enough SMs to use max_autotune_gemm mode


,слой,метод,мс на вызов,gpu_kernel_ms,kernels,syncs,memcpy,cpu_ops
0,"768→768, M=128",none,0.026,0.036,1.00,0.1,0.0,5.0
1,"768→768, M=128",none + compile,0.121,0.034,0.95,0.1,0.0,6.0
2,"768→768, M=128",torchao_int8_dyn,2.578,0.329,31.95,48.1,48.0,414.0
3,"768→768, M=128",torchao_int8_dyn + compile,0.304,0.053,2.95,0.1,0.0,8.0
4,"768→768, M=128",bnb_int8,0.527,0.107,5.95,2.1,2.0,36.0
5,"1024→3072, M=512",none,0.219,0.216,1.00,0.1,0.0,5.0
6,"1024→3072, M=512",none + compile,0.219,0.216,1.00,0.1,0.0,6.0
7,"1024→3072, M=512",torchao_int8_dyn,2.163,0.846,31.95,48.1,48.0,414.0
8,"1024→3072, M=512",torchao_int8_dyn + compile,0.272,0.194,2.95,0.1,0.0,8.0
9,"1024→3072, M=512",bnb_int8,0.531,0.306,5.95,2.1,2.0,36.0


In [8]:
# Что внутри одного квантованного слоя (eager): топ операций по времени CPU и по времени GPU
for method in PROFILE_METHODS[1:]:
    prof = layer_profiles.get((SHAPES[0][0], method, False))
    if prof is None:
        continue
    print(f"=================== {method}, слой {SHAPES[0][0]}→{SHAPES[0][1]}, M={SHAPES[0][2]} — по CPU")
    print(top_table(prof, "self_cpu_time_total", rows=10))
    print(f"=================== {method} — по GPU")
    print(top_table(prof, "self_device_time_total", rows=8))

=================== torchao_int8_dyn, слой 768→768, M=128 — по CPU
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg     Self CUDA   Self CUDA %    CUDA total  CUDA time avg    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                           quant::choose_qparams_affine        14.74%       7.984ms        35.19%      19.068ms     953.390us       0.000us         0.00%       1.153ms      57.646us            20  
                                        cudaMemcpyAsync        14.46%       7.834ms        14.46%       7.834ms       8.160us       0.000us 

### 8.2. Целая модель, один запрос

In [9]:
def model_case(name, method, compiled):
    spec = resolve_dtype(with_quant(load_spec(f"{REPO_DIR}/specs/models/{name}.yaml"), method), DEVICE)
    spec["compile"] = compiled
    adapter = make_adapter(spec, DEVICE)
    adapter.load()
    text = workloads(adapter)["legacy_medium"][0]
    try:
        for _ in range(WARMUP):
            adapter.predict([text])
        torch.cuda.synchronize()
        times = []
        for _ in range(N_REQUESTS):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            adapter.predict([text])
            torch.cuda.synchronize()
            times.append((time.perf_counter() - t0) * 1e3)
        with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
            for _ in range(N_REQUESTS):
                adapter.predict([text])
            torch.cuda.synchronize()
    finally:
        adapter.close()
    label = f"{name}__{method}{'__compile' if compiled else ''}"
    trace = f"{PROF_DIR}/{label}.json"
    prof.export_chrome_trace(trace)
    (open(f"{PROF_DIR}/{label}.txt", "w")).write(top_table(prof, "self_cpu_time_total", 25) + "\n" + top_table(prof, "self_device_time_total", 25))
    stats = trace_stats(trace, N_REQUESTS)
    wall = float(pd.Series(times).median())
    return {"модель": name, "метод": method + (" + compile" if compiled else ""), "p50 мс": round(wall, 1),
            "ядра GPU, мс": round(stats["gpu_kernel_ms"], 1), "GPU занята": f"{100 * stats['gpu_kernel_ms'] / wall:.0f}%",
            "ядер на запрос": round(stats["kernels"]), "синхронизаций": round(stats["syncs"], 1),
            "копирований": round(stats["memcpy"], 1), "CPU-операций": round(stats["cpu_ops"])}


model_rows = []
for name in PROFILE_MODELS:
    variants = [(m, False) for m in PROFILE_METHODS] + ([("torchao_int8_dyn", True)] if TRY_COMPILE else [])
    for method, compiled in variants:
        print(f"→ {name} / {method}{' + compile' if compiled else ''}")
        try:
            model_rows.append(model_case(name, method, compiled))
        except Exception as e:
            model_rows.append({"модель": name, "метод": method + (" + compile" if compiled else ""),
                               "ошибка": f"{type(e).__name__}: {e}"[:200]})
        free()

model_df = pd.DataFrame(model_rows)
model_df.to_csv(f"{PROF_DIR}/models.csv", index=False)
display(model_df)
print("трейсы и топ операций:", PROF_DIR)

→ koala / none


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer DebertaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1936 > 512). Running this sequence through the model will result in indexing errors


→ koala / torchao_int8_dyn


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1936 > 512). Running this sequence through the model will result in indexing errors


→ koala / bnb_int8


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1936 > 512). Running this sequence through the model will result in indexing errors


... (3 outputs left out)

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


→ hivetrace-guard-pro / torchao_int8_dyn
→ hivetrace-guard-pro / bnb_int8
→ hivetrace-guard-pro / torchao_int8_dyn + compile


,модель,метод,p50 мс,"ядра GPU, мс",GPU занята,ядер на запрос,синхронизаций,копирований,CPU-операций
0,koala,none,30.0,9.7,32%,714,5.1,17.0,3617
1,koala,torchao_int8_dyn,194.9,41.5,21%,2910,3461.1,3473.0,32849
2,koala,bnb_int8,82.6,18.7,23%,1457,185.1,233.0,7061
3,koala,torchao_int8_dyn + compile,45.7,8.7,19%,444,5.1,5.0,2555
4,hivetrace-guard-pro,none,101.1,94.0,93%,2215,5.1,5.0,8081
5,hivetrace-guard-pro,torchao_int8_dyn,473.9,194.6,41%,8093,9413.1,9413.0,87265
6,hivetrace-guard-pro,bnb_int8,253.4,118.8,47%,5205,580.1,763.0,19830
7,hivetrace-guard-pro,torchao_int8_dyn + compile,42.9,28.4,66%,555,4.1,4.0,2357


трейсы и топ операций: /kaggle/working/guard-eval/logs/profile


In [5]:
!cd {REPO_DIR} && python -m guard_eval compare --model hivetrace-guard-pro --a none --b torchao_int8_dyn \
    --set compile=true --limit-per-split 50 \
    --speed --speed-blocks 2 --speed-per-block 50 \
    2>&1 | grep -vE "Warning|warn\(|Loading weights|Fetching"
!cd {REPO_DIR} && python -m guard_eval summary && cat reports/SUMMARY.md

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
W1105 06:53:54.118000 1001 torch/_inductor/utils.py:1731] [0/0_1] Not enough SMs to use max_autotune_gemm mode
  merged = pd.concat([self.predictions(), new], ignore_index=True).drop_duplicates("text_hash", keep="last")
  merged = pd.concat([self.predictions(), new], ignore_index=True).drop_duplicates("text_hash", keep="last")
датасет: {'repo': 'AutotuneGuardrails/tails', 'revision': '8600f954140dc3b06e5a794a4f26e728c934838e', 'rows': 950, 'benchmarks': ['csrt', 'hivetrace_input', 'hivetrace_input_robust', 'multijail', 'rtplx']}
[hivetrace-guard-pro__none__float16] нужно посчитать 950 текстов → /kaggle/working/guard-eval/results/quality/runs/hivetrace-guard-pro/d16258ea9790
[hivetrace-guard-pro__none__float16] 512/950  (4.3 текстов/с)
[hivetrace-guard-pro__none__float16] 950/950  (6.9 текстов/с)
[hivetrace-guard-pro__torchao_int8_dyn__floa

In [6]:
!rm -f {REPO_DIR}/logs/profile/*.json